In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

import networkx as nx
import folium
import osmnx as ox
from matplotlib.pyplot import gray

In [2]:
def geocode_location(prompt_text):
    """
    :return: geo obj, text location from user input
    """
    while True:
        location = input(prompt_text)
        try:
            # TODO: Ensure location is within city limits
            geo = ox.geocode(location)
            return geo, location
        except Exception:
            print("Location not found. Please query again.")

# Source: ChatGPT
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2

    return 2 * R * math.asin(math.sqrt(a))

def loc_prompt():
    """
    :return: start and end geo locations
    """
    geo_start, loc_start = geocode_location("Please enter your starting location: ")
    geo_end, loc_end = geocode_location("Please enter your destination: ")

    print(f"Start Point: {geo_start} \'{loc_start}\'")
    print(f"End Point: {geo_end} \'{loc_end}\'")

    points = [geo_start, geo_end]
    return points

def center(points):
    """
    :param points:
    :return: center point and the correct radius for a plot
    """
    start_lat = points[0][0]
    start_lon = points[0][1]
    end_lat = points[1][0]
    end_lon = points[1][1]

    geo_center = (
        ( float(start_lat)+float(end_lat) ) / 2,
        ( float(start_lon)+float(end_lon) ) / 2
                 )

    # Haversine distance between locations * 1.25 for some bezels for mapping
    bezel_factor = 1.25
    geo_dist = (haversine(start_lat, start_lon, end_lat, end_lon) / 2) * bezel_factor
    # geo_dist = max(geo_dist, 1000)
    geo_dist = round(geo_dist, 2)

    return geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon

In [3]:
G = ox.graph_from_place("Washington, District of Columbia, USA", network_type="walk") # ~2 min runtime

Crime density weight


In [5]:
crime_gdf = gpd.read_file("../../Data/Crime/Crime_Incidents_in_2025.geojson")

In [6]:
graph = ox.project_graph(G)  # projects to UTM automatically
edges = ox.graph_to_gdfs(graph, nodes=False, edges=True)

crime_gdf = crime_gdf.to_crs(edges.crs)

In [7]:
buffer_dist = 50  # HYPERPARAM: buffer distance in meters
                  # TODO: Incorperate into function to allow user to dial in

edges['geometry_buffer'] = edges.geometry.buffer(buffer_dist)

edges_buffered = edges.set_geometry('geometry_buffer')

# spatial join: crimes within each edge buffer
joined = gpd.sjoin(crime_gdf, edges_buffered, how='left', predicate='within')

In [8]:
crime_counts = joined.groupby(['u', 'v', 'key']).size()

edges['crime_count'] = edges.index.map(crime_counts).fillna(0)
edges['crime_density'] = edges['crime_count'] / edges['length']

In [10]:
alpha = 10  # HYPERPARAM: tune this (0 -> 10+)
            # TODO: Incorperate into function to allow user to dial in

edges['crime_density_norm'] = (edges['crime_density'] / edges['crime_density'].max())

edges['crime_weight'] = edges['length'] * (1 + alpha * edges['crime_density_norm'])

In [11]:
for u, v, k, data in graph.edges(keys=True, data=True):
    if (u, v, k) in edges.index:
        data['crime_weight'] = edges.loc[(u, v, k), 'crime_weight']
    else:
        print('WARING: fallback to length occurred')
        data['crime_weight'] = data['length']  # fallback

User


In [ ]:
# Functions have built in print statements
geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon = center(loc_prompt())

In [ ]:
graph = ox.graph_from_point(geo_center, dist=geo_dist, network_type='walk') # runtime dependant of graph size

Folium graph


In [ ]:
from folium.plugins import *

m = folium.Map(
    location = geo_center,
    zoom_start = 15

)

In [ ]:
edges_gdf = ox.graph_to_gdfs(graph, nodes=False, edges=True)


In [ ]:
for _, row in edges_gdf.iterrows():
    coords = [(lat, lon) for lon, lat in row['geometry'].coords]
    folium.PolyLine(coords, color = '#888888', weight = 1, opacity = 0.5).add_to(m)

In [ ]:
route_coords = [
    (graph.nodes[node]['y'], graph.nodes[node]['x'])
    for node in route
]

In [ ]:
ScrollZoomToggler().add_to(m)
folium.PolyLine(route_coords, color = 'red', weight = 4, opacity = 0.75).add_to(m)
folium.Marker(
    location = origin_xy,
    icon = folium.Icon(color='green', icon = 'flag', prefix='fa')
).add_to(m)
folium.Marker(
    location = destination_xy,
    icon = folium.Icon(color='red', icon = 'flag', prefix = 'fa')
).add_to(m)

m